# MDS — ILP (PuLP)

$$\min \sum_v x_v \quad s.t. \quad \forall v:\; \sum_{u\in N[v]} x_u \ge 1,\; x_v\in\{0,1\}$$

Solver CBC vía PuLP. Baseline **clásico exacto** (poliédrico).

> Dependencia: `pulp==2.9.0` recomendado (CBC incluido). Compatible también con PuLP 4.x.


In [1]:
import itertools
import json
import math
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np

import pulp


In [2]:
nodes = ["Mario", "Sarah", "Raul", "Ana", "Enrique", "Saul"]
edges = [("Mario", "Sarah"), ("Mario", "Raul"), ("Mario", "Ana"),
         ("Enrique", "Sarah"), ("Enrique", "Raul"), ("Saul", "Ana")]
idx = {v: i for i, v in enumerate(nodes)}
n = len(nodes)
adj = {i: set() for i in range(n)}
for a, b in edges:
    adj[idx[a]].add(idx[b])
    adj[idx[b]].add(idx[a])
closed = {v: sorted(adj[v] | {v}) for v in range(n)}

def bits_cost_payments(bits):
    return int(sum(bits))

def is_dominating(bits):
    return all(any(bits[u] for u in closed[v]) for v in range(n))

def decode(bits):
    return [nodes[i] for i in range(n) if bits[i]]

# Óptimo exacto (referencia compartida)
best_cost, best_bits = n + 1, None
for k in range(2 ** n):
    bits = [(k >> i) & 1 for i in range(n)]
    if is_dominating(bits):
        c = bits_cost_payments(bits)
        if c < best_cost:
            best_cost, best_bits = c, bits
OPTIMAL_COST = float(best_cost)
OPTIMAL_SOL = decode(best_bits)
print("Óptimo MDS:", OPTIMAL_SOL, "coste =", OPTIMAL_COST)

def resolve_metrics_dir() -> Path:
    for base in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        root = base / "Notebooks" / "Optimization" / "MinimumDominatingSetOptimization"
        if root.is_dir():
            d = root / "metrics"
            d.mkdir(parents=True, exist_ok=True)
            return d
    d = Path.cwd().resolve() / "Notebooks" / "Optimization" / "MinimumDominatingSetOptimization" / "metrics"
    d.mkdir(parents=True, exist_ok=True)
    return d

def save_mds_metrics(payload: dict, filename: str):
    path = resolve_metrics_dir() / filename
    path.write_text(json.dumps(payload, indent=2))
    print("Saved:", path)
    return path


Óptimo MDS: ['Ana', 'Enrique'] coste = 2.0


## Modelo y resolución


In [3]:
# Compatibilidad PuLP 2.x (dicts + CBC embebido) y 4.x (add_variable_*)
import pulp

def add_binary_dicts(prob, name, indices):
    if hasattr(prob, "add_variable_dicts"):
        return prob.add_variable_dicts(name, indices, cat="Binary")
    return pulp.LpVariable.dicts(name, indices, cat="Binary")

def add_binary(prob, name):
    if hasattr(prob, "add_variable"):
        return prob.add_variable(name, cat="Binary")
    return pulp.LpVariable(name, cat="Binary")

def solve_cbc(prob):
    if hasattr(pulp, "PULP_CBC_CMD"):
        return prob.solve(pulp.PULP_CBC_CMD(msg=False))
    available = pulp.listSolvers(onlyAvailable=True)
    if not available:
        raise RuntimeError(
            "No hay solver PuLP disponible. Recomendado: pip install 'pulp==2.9.0' "
            "(incluye CBC)."
        )
    return prob.solve(pulp.getSolver(available[0], msg=0))

def status_label(status):
    code = getattr(status, "status", status)
    try:
        return pulp.LpStatus[code]
    except Exception:
        return str(code)

def var_value(v):
    if hasattr(v, "value"):
        val = v.value()
        if val is not None:
            return float(val)
    return float(pulp.value(v))

t0 = time.perf_counter()
prob = pulp.LpProblem("MDS", pulp.LpMinimize)
x = add_binary_dicts(prob, "x", list(range(n)))
prob += pulp.lpSum(x[i] for i in range(n))
for v in range(n):
    prob += pulp.lpSum(x[u] for u in closed[v]) >= 1
status = solve_cbc(prob)
elapsed = time.perf_counter() - t0
bits = [1 if var_value(x[i]) > 0.5 else 0 for i in range(n)]
sol = decode(bits)
c = bits_cost_payments(bits)
feasible = is_dominating(bits)
success = feasible and abs(c - OPTIMAL_COST) < 1e-9
print("Status:", status_label(status), "sol:", sol, "coste:", c)

payload = {
    "problem": "mds",
    "solver": "ilp",
    "family": "classical",
    "protocol": {"n_runs": 1, "method": "pulp_cbc_covering_ilp", "pulp_version": pulp.__version__},
    "instance": {"nodes": nodes, "edges": [list(e) for e in edges]},
    "quality": {
        "optimal_cost": OPTIMAL_COST,
        "optimal_solution": OPTIMAL_SOL,
        "cost_mean": float(c),
        "cost_best": float(c),
        "gap_mean": float(c - OPTIMAL_COST),
    },
    "success_rate": float(success),
    "time_s": {"mean": float(elapsed), "std": 0.0, "total": float(elapsed)},
    "effort": {"milp_status": status_label(status), "n_binary_vars": n},
    "runs": [{"seed": 0, "solution": sol, "cost": float(c), "feasible": feasible, "success": bool(success), "time_s": float(elapsed)}],
}
save_mds_metrics(payload, "ilp_mds.json")


Status: Optimal sol: ['Ana', 'Enrique'] coste: 2
Saved: /Users/juanprada/Documents/UNIR-MasterQC-QuantumAdvantageInDataScience/Notebooks/Optimization/MinimumDominatingSetOptimization/metrics/ilp_mds.json


PosixPath('/Users/juanprada/Documents/UNIR-MasterQC-QuantumAdvantageInDataScience/Notebooks/Optimization/MinimumDominatingSetOptimization/metrics/ilp_mds.json')